Test LSTM Architecture for Rebuilding Joint Angles from Current, Speed, and Temperature Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_feather('../data/aursad/aursad_training.feather')

In [ ]:
df.head()

In [ ]:
from ikpy.chain import Chain
from ikpy.link import URDFLink, OriginLink
import numpy as np

# --- Define the UR3 robot arm chain ---
UR3_arm = Chain(name="UR3_arm", links=[
    # OriginLink(),  # Base

    # Shoulder Pan
    URDFLink(
        name="shoulder_pan",
        origin_translation=[0, 0, 0.15185],
        origin_orientation=[0, 0, 0],
        rotation=[0, 0, 1],
        # bounds=(-1.65, 1.0)
        # bounds=(-bounds, bounds)
    ),

    # Shoulder Lift
    URDFLink(
        name="shoulder_lift",
        origin_translation=[0, 0.1197, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-6, -0.0),
        # bounds=(-bounds, bounds)
    ),

    # Elbow
    URDFLink(
        name="elbow",
        origin_translation=[0.24365, 0, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-2.5, 2.5),
        # bounds=(-bounds, bounds)
    ),

    # Wrist 1
    URDFLink(
        name="wrist_1",
        origin_translation=[0.21325, 0, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-0.8, 4.0),
        # bounds=(-bounds, bounds)
    ),

    # Wrist 2
    URDFLink(
        name="wrist_2",
        origin_translation=[0, 0.08535, 0],
        origin_orientation=[0, 0, 0],
        rotation=[0, 0, 1],
        # bounds=(-2.5, 3.0),
        # bounds=(-bounds, bounds)
    ),

    # Wrist 3 / End Effector
    URDFLink(
        name="wrist_3",
        origin_translation=[0, 0, 0.0819],
        origin_orientation=[0, 0, 0],
        rotation=[0, 1, 0],
        # bounds=(-1.7, -1.5)
        # bounds=(-bounds, bounds)

    )
])

df_angles = df[['q0', 'q1', 'q2', 'q3', 'q4', 'q5']]
animation_sequence = df_angles.values.tolist()

pos_lst = []
for frame in animation_sequence:
    pos = UR3_arm.forward_kinematics(frame)[3,:3]
    pos_lst.append(pos)
    
df = pd.concat([df.reset_index(drop=True), pd.DataFrame(pos_lst, columns=['x', 'y', 'z']).reset_index(drop=True)], axis=1)
df.head()

In [ ]:
# df.to_feather('../data/aursad/aursad_training.feather')

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Model Definition
# -----------------------------
class RobotLSTM(nn.Module):
    def __init__(self, input_size=18, hidden_size=128, output_size=6, num_layers=2):
        super(RobotLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out)
        return out

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd


df = pd.read_feather('../data/aursad/aursad_training.feather')

# -----------------------------
# Normalization Setup
# -----------------------------
input_scaler = StandardScaler()
output_scaler = StandardScaler()

input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z']
type_name = 'Current'
output_lst = [f'{type_name}{i}' for i in range(6)]
input_size = len(input_lst)
output_size = len(output_lst)
seq_len = 50

# Fit on all data (you can also fit only on train split for better generalization)
X_scaled = input_scaler.fit_transform(df[input_lst])
y_scaled = output_scaler.fit_transform(df[output_lst])

# -----------------------------
# Reshape to Sequences
# -----------------------------
num_samples = len(X_scaled) // seq_len
X = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)
y = y_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, output_size)

# Convert to float tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

In [ ]:
# -----------------------------
# AURSAD Data
# -----------------------------

dataset = TensorDataset(X_tensor, y_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

# -----------------------------
# Training Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RobotLSTM(input_size, 128, output_size, 2).to(device)
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)


# -----------------------------
# Training Loop
# -----------------------------
epochs = 100
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        # scheduler.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            preds = model(X_val)
            val_loss += criterion(preds, y_val).item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "lstm_robot_model.pt")
        print("Saved best model")

print("Training complete. Best validation loss:", best_val_loss)

# -----------------------------
# Plot Learning Curves
# -----------------------------
plt.figure(figsize=(8,5))
plt.plot(train_losses, label='Training Loss', color='blue', linewidth=2)
plt.plot(val_losses, label='Validation Loss', color='orange', linewidth=2)
plt.title('LSTM Training and Validation Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# -----------------------------
# Load Model for Inference
# -----------------------------
loaded_model = RobotLSTM(input_size, 128, output_size, 2)
loaded_model.load_state_dict(torch.load("lstm_robot_model.pt", map_location=device))
loaded_model.to(device)
loaded_model.eval()

with torch.no_grad():
    test_input = torch.randn(1, seq_len, input_size).to(device)
    pred_angles = loaded_model(test_input)
    print("Predicted joint angles shape:", pred_angles.shape)


In [ ]:
# -----------------------------
# Load Model for Inference
# -----------------------------
loaded_model = RobotLSTM(input_size, 128, output_size, 2)
loaded_model.load_state_dict(torch.load("lstm_robot_model.pt", map_location=device))
loaded_model.to(device)
loaded_model.eval()

with torch.no_grad():
    test_input = torch.randn(1, seq_len, input_size).to(device)
    pred_angles = loaded_model(test_input)
    print("Predicted joint angles shape:", pred_angles.shape)

In [ ]:
# seq_len = 50
# input_size = 9
# output_size = 18

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# loaded_model = RobotLSTM(input_size, 128, output_size, 2)
# loaded_model.load_state_dict(torch.load("lstm_robot_model.pt", map_location=device))
# loaded_model.to(device)
# loaded_model.eval()

# X = df[['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z']].values

# pred_lst = ['Speed0', 'Speed1', 'Speed2', 'Speed3', 'Speed4', 'Speed5', 
#          'Temperature0','Temperature1','Temperature2','Temperature3','Temperature4','Temperature5',
#          'Current0','Current1','Current2','Current3','Current4','Current5']
# y =  df[pred_lst].values
# num_samples = len(X) // seq_len

# -----------------------------
# Load Model for Full Dataset Inference
# -----------------------------
# loaded_model = RobotLSTM(input_size, 128, output_size, 2)
# loaded_model.load_state_dict(torch.load("lstm_robot_model.pt", map_location=device))
# loaded_model.to(device)
# loaded_model.eval()

# -----------------------------
# Prepare Data as Sequential Input
# -----------------------------
# Ensure data is reshaped as (num_samples, seq_len, input_size)
num_samples = len(X_scaled) // seq_len
X_seq = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)

# Convert to tensor
X_seq_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)

# -----------------------------
# Run Inference 
# -----------------------------
predictions_scaled = []

with torch.no_grad():
    for batch in X_seq_tensor:
        batch = batch.unsqueeze(0)  # shape (1, seq_len, input_size)
        pred = loaded_model(batch)  # shape (1, seq_len, output_size)
        predictions_scaled.append(pred.squeeze(0).cpu().numpy())

# Stack all batches into one (num_samples * seq_len, output_size)
predictions_scaled = np.vstack(predictions_scaled)

# -----------------------------
# Denormalize Predictions
# -----------------------------
predictions_original = output_scaler.inverse_transform(predictions_scaled)


# -----------------------------
# Save Predictions to DataFrame
# -----------------------------
pred_cols = [f'pred_{name}' for name in output_lst]
pred_df = pd.DataFrame(predictions_original, columns=pred_cols)

# Trim df to match predictions (in case of leftover rows)
df_pred = df.iloc[:len(pred_df)].copy()
df_pred[pred_cols] = pred_df

In [ ]:
# -----------------------------
# Save to file
# -----------------------------
df_pred.to_feather("pred_speed.feather")

print("Predictions complete")
print("Saved file")
print("DataFrame shape:", df_pred.shape)
print(df_pred.head())

In [2]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def time_series_plots(df, feature_type_lst, error=None):
    
    colors = px.colors.qualitative.Dark24[:6]

    # Create subplots for X, Y, Z
    fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=True,
        subplot_titles=(feature_type_lst),
        vertical_spacing=0.08
    )

    # Loop through each axis
    for idx, feature_type in enumerate(feature_type_lst, start=1):
        color = colors[idx - 1]

        # Plot original position
        if feature_type in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[feature_type],
                    name=f"{feature_type.upper()} (true)",
                    mode='lines',
                    line=dict(color=color, width=2),
                ),
                row=idx, col=1
            )

        # Plot reconstructed / predicted position
        hat_col = f"pred_{feature_type}"
        if hat_col in df.columns:
            fig.add_trace(
                go.Scatter(
                    x=df['time'],
                    y=df[hat_col],
                    name=f"{feature_type.upper()} (pred)",
                    mode='lines',
                    line=dict(color=color, width=2, dash='dash')
                ),
                row=idx, col=1
            )

    # Optional: error shading or line
    if error is not None and "time" in df.columns:
        fig.add_trace(
            go.Scatter(
                x=df['time'],
                y=error,
                name="Error",
                mode='lines',
                line=dict(color='red', dash='dot')
            ),
            row=3, col=1
        )

    # Axis titles
    fig.update_xaxes(title_text="Time (s)", row=6, col=1, rangeslider_visible=True)
    for i, feature_type in enumerate(feature_type_lst, start=1):
        fig.update_yaxes(title_text=f"{feature_type.upper()} (m)", row=i, col=1)

    # Layout
    fig.update_layout(
        height=1000,
        title_text="Feature Reconstruction via LSTM",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1)
    )

    return fig

# Example usage
# fig = time_series_plots(df_pred.iloc[::100], output_lst)
# fig.show()

In [ ]:
import torch
import pandas as pd


df_rad = pd.read_feather('../data/rad/rad.feather')

# -----------------------------
# Normalization Setup
# -----------------------------
input_scaler = StandardScaler()
output_scaler = StandardScaler()

input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z']
type_name = 'Current'
output_lst = [f'{type_name}{i}' for i in range(6)]
input_size = len(input_lst)
output_size = len(output_lst)
seq_len = 50

# Fit on all data (you can also fit only on train split for better generalization)
X_scaled = input_scaler.fit_transform(df[input_lst])
y_scaled = output_scaler.fit_transform(df[output_lst])

# -----------------------------
# Reshape to Sequences
# -----------------------------
num_samples = len(X_scaled) // seq_len
X = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)
y = y_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, output_size)

# Convert to float tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

# -----------------------------
# Run Inference on All Sequences
# -----------------------------
predictions = []

with torch.no_grad():
    for batch in X_seq_tensor:
        batch = batch.unsqueeze(0)  # shape (1, seq_len, input_size)
        pred = loaded_model(batch)  # shape (1, seq_len, output_size)
        predictions.append(pred.squeeze(0).cpu().numpy())

# Convert to single 2D array (total_timesteps, output_size)
predictions = np.vstack(predictions)
predictions_original = output_scaler.inverse_transform(predictions_scaled)

# -----------------------------
# Save Predictions to DataFrame
# -----------------------------
pred_cols = output_lst
pred_df = pd.DataFrame(predictions_original, columns=pred_cols)

# Trim df to match predictions (in case of leftover rows)
df_pred = df_rad.iloc[:len(pred_df)].copy()
df_pred[pred_cols] = pred_df

In [ ]:
df_pred.head()

In [ ]:
fig = time_series_plots(df_pred, feature_type_lst=output_lst)
fig.show()

In [ ]:
# Ensure both DataFrames have the same index length
df_merged = pd.concat([df_rad.reset_index(drop=True),
                       df_pred.reset_index(drop=True)],
                      axis=1)

# Remove any duplicate columns that might appear
df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]

print("Combined DataFrame created with all unique columns.")
print("Final shape:", df_merged.shape)
print("Sample columns:", df_merged.columns.tolist())

# save
df_merged.to_feather("../data/rad/rad_add_current.feather")


In [3]:
from sklearn.preprocessing import StandardScaler
import pandas as pd


# df = pd.read_feather('../data/aursad/aursad_training.feather')
input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z', 
            'Current0', 'Current1', 'Current2', 'Current3', 'Current4', 'Current5']
output_type_name = 'Speed'

def data_prep(df, input_lst, output_type_name, is_target_known=True):
    # -----------------------------
    # Normalization Setup
    # -----------------------------
    input_scaler = StandardScaler()
    output_scaler = None

    output_lst = [f'{output_type_name}{i}' for i in range(6)]
    input_size = len(input_lst)
    output_size = len(output_lst)
    seq_len = 50

    # Fit on all data (you can also fit only on train split for better generalization)
    X_scaled = input_scaler.fit_transform(df[input_lst])
    num_samples = len(X_scaled) // seq_len

    y_tensor = None
    if is_target_known:
        output_scaler = StandardScaler()
        y_scaled = output_scaler.fit_transform(df[output_lst])
        y = y_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, output_size)
        y_tensor = torch.tensor(y, dtype=torch.float32)

    # -----------------------------
    # Reshape to Sequences
    # -----------------------------
    X = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)

    # Convert to float tensors
    X_tensor = torch.tensor(X, dtype=torch.float32)

    return X_scaled, X_tensor, y_tensor, output_scaler

# X_tensor, y_tensor, output_scaler = data_prep(df, input_lst, output_type_name, output_scaler)

In [4]:
# -----------------------------
# AURSAD Data
# -----------------------------
def train_lstm(X_tensor, y_tensor, epochs, model_name):
    dataset = TensorDataset(X_tensor, y_tensor)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    # -----------------------------
    # Training Setup
    # -----------------------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = RobotLSTM(input_size, 128, output_size, 2).to(device)
    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)


    # -----------------------------
    # Training Loop
    # -----------------------------
    best_val_loss = float('inf')
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            # scheduler.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                preds = model(X_val)
                val_loss += criterion(preds, y_val).item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), model_name)
            print("Saved best model")

    print("Training complete. Best validation loss:", best_val_loss)

    # -----------------------------
    # Plot Learning Curves
    # -----------------------------
    plt.figure(figsize=(8,5))
    plt.plot(train_losses, label='Training Loss', color='blue', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', color='orange', linewidth=2)
    plt.title('LSTM Training and Validation Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# train_lstm(X_tensor, y_tensor, 30, "speed_pred_lstm30")

In [4]:
def prediction_df(X_scaled, output_scaler, seq_len, input_size, output_size, model_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaded_model = RobotLSTM(input_size, 128, output_size, 2)
    loaded_model.load_state_dict(torch.load(model_name, map_location=device))
    loaded_model.to(device)
    loaded_model.eval()

    # -----------------------------
    # Prepare Data as Sequential Input
    # -----------------------------

    # Ensure data is reshaped as (num_samples, seq_len, input_size)
    num_samples = len(X_scaled) // seq_len
    X_seq = X_scaled[:num_samples * seq_len].reshape(num_samples, seq_len, input_size)

    # Convert to tensor
    X_seq_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)

    # -----------------------------
    # Run Inference 
    # -----------------------------
    predictions_scaled = []

    with torch.no_grad():
        for batch in X_seq_tensor:
            batch = batch.unsqueeze(0)  # shape (1, seq_len, input_size)
            pred = loaded_model(batch)  # shape (1, seq_len, output_size)
            predictions_scaled.append(pred.squeeze(0).cpu().numpy())

    # Stack all batches into one (num_samples * seq_len, output_size)
    predictions_scaled = np.vstack(predictions_scaled)

    # -----------------------------
    # Denormalize Predictions
    # -----------------------------
    predictions_original = output_scaler.inverse_transform(predictions_scaled)

    # -----------------------------
    # Save Predictions to DataFrame
    # -----------------------------
    pred_cols = [f'pred_{name}' for name in output_lst]
    pred_df = pd.DataFrame(predictions_original, columns=pred_cols)

    # Trim df to match predictions (in case of leftover rows)
    # df_pred = df.iloc[:len(pred_df)].copy()
    # df_pred[pred_cols] = pred_df

    return pred_df

# df_pred = prediction_df(X_scaled, output_scaler)
# fig = time_series_plots(df_pred.iloc[::100], output_lst)
# fig.show()

In [5]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import torch

df_aursad = pd.read_feather('../data/aursad/aursad_training.feather')
df_rad = pd.read_feather('../data/rad/rad_add_current.feather')
input_lst = ['q0','q1','q2','q3','q4','q5', 'x', 'y', 'z', 
            'Current0', 'Current1', 'Current2', 'Current3', 'Current4', 'Current5']
output_type_name = 'Speed'
output_lst = [f'{output_type_name}{i}' for i in range(6)]

X_scaled, X_tensor, _, _ = data_prep(df_rad, input_lst, output_type_name, is_target_known=False)

output_scaler = StandardScaler()
y_scaled = output_scaler.fit_transform(df_aursad[output_lst])

df_pred = prediction_df(X_scaled, output_scaler, 50, 
                        len(input_lst), len(output_lst), 
                        '../models/speed_pred_lstm30.pt')
df_pred = pd.concat([df_pred.reset_index(drop=True), df_rad['time'].reset_index(drop=True)], axis=1)

output_lst = [f'pred_{output_type_name}{i}' for i in range(6)]
fig = time_series_plots(df_pred.iloc[::100], output_lst)
fig.show()

In [6]:
# Ensure both DataFrames have the same index length
df_merged = pd.concat([df_rad.reset_index(drop=True),
                       df_pred.reset_index(drop=True)],
                      axis=1)

# Remove any duplicate columns that might appear
df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]

print("Combined DataFrame created with all unique columns.")
print("Final shape:", df_merged.shape)
print("Sample columns:", df_merged.columns.tolist())

# save
df_merged.to_feather("../data/rad/rad_current_speed.feather")
df_merged

Combined DataFrame created with all unique columns.
Final shape: (11103, 22)
Sample columns: ['time', 'x', 'y', 'z', 'q0', 'q1', 'q2', 'q3', 'q4', 'q5', 'Current0', 'Current1', 'Current2', 'Current3', 'Current4', 'Current5', 'pred_Speed0', 'pred_Speed1', 'pred_Speed2', 'pred_Speed3', 'pred_Speed4', 'pred_Speed5']


,time,x,y,z,q0,q1,q2,q3,q4,q5,...,Current2,Current3,Current4,Current5,pred_Speed0,pred_Speed1,pred_Speed2,pred_Speed3,pred_Speed4,pred_Speed5
0,0.000000,-0.186053,-0.013900,0.25,1.452211,-0.934004,-2.250000,-0.307446,0.0,0.0,...,-1.365911,0.064612,-0.049508,-0.032210,0.255485,0.022222,-0.025700,-0.007818,-0.023141,-0.174272
1,0.846160,-0.181287,-0.004773,0.25,1.439431,-0.866526,-2.250000,-0.307445,0.0,0.0,...,-1.334094,0.079111,-0.054991,-0.038373,0.316733,0.024348,0.007687,-0.010320,-0.015131,-0.200029
2,1.290711,-0.174021,0.002766,0.25,1.479069,-0.717666,-2.250000,-0.307430,0.0,0.0,...,-1.358874,0.079595,-0.049325,-0.033965,0.330808,0.046861,0.028330,-0.011054,-0.015320,-0.236209
3,1.717405,-0.165193,0.008004,0.25,1.522477,-0.583960,-2.250000,-0.307440,0.0,0.0,...,-1.342593,0.082664,-0.020003,-0.015914,0.368647,0.074818,0.045840,-0.011660,-0.014686,-0.282943
4,2.163429,-0.156365,0.013004,0.25,1.487843,-0.584043,-2.250000,-0.307368,0.0,0.0,...,-1.306100,0.077683,-0.004220,-0.010779,0.386458,0.079184,0.036964,-0.010259,-0.011356,-0.308225
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11098,16532.287049,-0.154900,0.212000,0.25,1.305599,0.324787,-1.867403,-0.500559,0.0,0.0,...,-1.114726,0.075522,0.024877,0.033898,0.084597,0.027688,-0.099336,-0.008574,0.027155,-0.272748
11099,16532.646601,-0.144900,0.212714,0.25,1.246993,0.259364,-1.908266,-0.609200,0.0,0.0,...,-1.180680,0.052444,0.017368,0.027123,0.118920,0.029704,-0.092126,-0.006197,0.013080,-0.239671
11100,16533.064527,-0.134900,0.213429,0.25,1.186681,0.249422,-1.945261,-0.583451,0.0,0.0,...,-0.628255,0.386206,0.058247,-0.020473,NaN,NaN,NaN,NaN,NaN,NaN
11101,16533.480524,-0.124900,0.214619,0.25,1.126301,0.239886,-1.976885,-0.558166,0.0,0.0,...,-0.946629,0.166760,-0.048340,0.077496,NaN,NaN,NaN,NaN,NaN,NaN
